# Optuna Hyperparameter Search — PCOS GNN Framework
**Thesis:** A Machine Learning Framework for Hormonal and Metabolic Subtype Discovery  
and Classification in PCOS Using Tabular Clinical Data

## Protocol
This notebook implements a **nested hyperparameter optimisation** protocol:

1. The outer 80/20 train/test split is reconstructed with the same seed as all other notebooks.  
   The 20% test set is **never accessed here**.
2. From the 80% training pool, a **single inner 80/20 stratified split** is created as the  
   Optuna validation surface.
3. Optuna runs **50 trials per model**, optimising **inner-validation MCC** (primary thesis metric).
4. Best hyperparameters per model are saved to `best_hyperparams.json`.
5. The five main CV notebooks are then re-run with per-model configs from that JSON.

**Fixed parameters (not searched):**  
`K_NEIGHBOURS=10`, `N2V_DIM=64`, `N2V_WALK_LEN=20`, `N2V_CONTEXT=10`,  
`N2V_WALKS=10`, `N2V_EPOCHS=50`, `N2V_LR=0.01`  
These are held constant because (a) they are structural rather than architectural  
and (b) searching them would multiply the trial cost by 3–5×.


In [9]:
# CELL 1 — INSTALLATION
# Run this cell first. Restart the kernel after it completes.

import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

pip_install('torch_geometric')
pip_install('imbalanced-learn')
pip_install('optuna')

print("Installation complete. Restart the kernel, then run Cell 2 onwards.")


Installation complete. Restart the kernel, then run Cell 2 onwards.


In [10]:
# CELL 2 — IMPORTS, SEED, FIXED CONFIGURATION

import os, gc, json, random, warnings
warnings.filterwarnings('ignore')
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)   # suppress per-trial noise

import numpy  as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.nn import (Linear, Sequential, BatchNorm1d, ReLU)

import torch_geometric
from torch_geometric.data  import Data
from torch_geometric.nn    import (GCNConv, GATConv, SAGEConv, GINConv, NNConv)
from torch_geometric.utils import coalesce
from gensim.models         import Word2Vec

from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import StandardScaler
from sklearn.metrics.pairwise  import cosine_similarity
from sklearn.metrics           import matthews_corrcoef, roc_auc_score
from imblearn.over_sampling    import SMOTE

print(f"PyTorch           : {torch.__version__}")
print(f"PyTorch Geometric : {torch_geometric.__version__}")
print(f"Optuna            : {optuna.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

GLOBAL_SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(GLOBAL_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device            : {DEVICE}")

# ── Fixed configuration (identical to all CV notebooks) ──────────
BASE_CONFIG = {
    'DATA_PATH'    : ('/kaggle/input/datasets/monamehrun/'
                      'pcos-cleaned-dataset/pcos_cleaned.csv'),
    'TARGET_COL'   : 'PCOS',
    'OUTPUT_DIR'   : '/kaggle/working/',
    'TEST_SIZE'    : 0.20,          # outer split — test set sealed
    'INNER_VAL'    : 0.20,          # inner split for Optuna
    'SEED'         : GLOBAL_SEED,
    'K_NEIGHBOURS' : 10,
    'N2V_DIM'      : 64,
    'N2V_WALK_LEN' : 20,
    'N2V_CONTEXT'  : 10,
    'N2V_WALKS'    : 10,
    'N2V_EPOCHS'   : 50,
    'N2V_LR'       : 0.01,
    # Training budget per Optuna trial (reduced to keep search feasible)
    'TRIAL_EPOCHS' : 150,
    'PATIENCE'     : 30,
    # Number of Optuna trials per model
    'N_TRIALS'     : 100,
}

print("\nBase configuration loaded.")
print(f"Trial epochs   : {BASE_CONFIG['TRIAL_EPOCHS']} (with patience {BASE_CONFIG['PATIENCE']})")
print(f"Trials/model   : {BASE_CONFIG['N_TRIALS']}")


PyTorch           : 2.10.0+cu128
PyTorch Geometric : 2.8.0
Optuna            : 4.8.0
CUDA available    : True
Device            : cuda

Base configuration loaded.
Trial epochs   : 150 (with patience 30)
Trials/model   : 100


In [11]:
# CELL 3 — GRAPH, NODE2VEC AND SMOTE UTILITIES
# Exact copies from the five CV notebooks. Do not modify.

def build_knn_graph(features_scaled: np.ndarray, k: int = 10):
    n   = len(features_scaled)
    sim = cosine_similarity(features_scaled)
    np.fill_diagonal(sim, -2.0)
    src, dst, wts = [], [], []
    for i in range(n):
        top_k = np.argpartition(sim[i], -k)[-k:]
        for j in top_k:
            w = float(max(0.0, sim[i][j]))
            src += [i, j];  dst += [j, i];  wts += [w, w]
    ei = torch.tensor([src, dst], dtype=torch.long)
    ew = torch.tensor(wts,        dtype=torch.float)
    ei, ew = coalesce(ei, ew, num_nodes=n, reduce='max')
    return ei, ew


def add_synthetic_nodes(ei, ew, X_real_sc, X_syn_sc, k=10):
    n_real, n_syn = len(X_real_sc), len(X_syn_sc)
    if n_syn == 0:
        return ei, ew
    sim = cosine_similarity(X_syn_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_syn):
        s = n_real + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [s, j];  dst += [j, s];  wts += [w, w]
    new_ei = torch.tensor([src, dst], dtype=torch.long)
    new_ew = torch.tensor(wts,        dtype=torch.float)
    aug_ei = torch.cat([ei, new_ei], dim=1)
    aug_ew = torch.cat([ew, new_ew])
    aug_ei, aug_ew = coalesce(aug_ei, aug_ew,
                               num_nodes=n_real + n_syn, reduce='max')
    return aug_ei, aug_ew


def add_val_nodes(ei_aug, ew_aug, X_real_sc, X_val_sc, k=10, n_train_aug=None):
    n_real = len(X_real_sc)
    n_val  = len(X_val_sc)
    if n_train_aug is None:
        n_train_aug = n_real
    sim = cosine_similarity(X_val_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_val):
        v = n_train_aug + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [v, j];  dst += [j, v];  wts += [w, w]
    new_ei  = torch.tensor([src, dst], dtype=torch.long)
    new_ew  = torch.tensor(wts,        dtype=torch.float)
    comb_ei = torch.cat([ei_aug, new_ei],  dim=1)
    comb_ew = torch.cat([ew_aug, new_ew])
    return comb_ei, comb_ew


def _random_walks(edge_index, num_nodes, walk_length, walks_per_node, seed=42):
    import random as _r
    _r.seed(seed)
    adj = [[] for _ in range(num_nodes)]
    ei  = edge_index.cpu().numpy()
    for s, d in zip(ei[0], ei[1]):
        adj[int(s)].append(int(d))
    walks = []
    nodes = list(range(num_nodes))
    for _ in range(walks_per_node):
        _r.shuffle(nodes)
        for start in nodes:
            walk = [start]
            for _ in range(walk_length - 1):
                curr = walk[-1]
                nbrs = adj[curr]
                if nbrs:
                    walk.append(_r.choice(nbrs))
                else:
                    break
            walks.append([str(n) for n in walk])
    return walks


def train_node2vec(edge_index, num_nodes: int, cfg: dict, device):
    walks = _random_walks(edge_index, num_nodes,
                          walk_length    = cfg['N2V_WALK_LEN'],
                          walks_per_node = cfg['N2V_WALKS'],
                          seed           = cfg['SEED'])
    w2v = Word2Vec(sentences   = walks,
                   vector_size = cfg['N2V_DIM'],
                   window      = cfg['N2V_CONTEXT'],
                   min_count   = 0,
                   sg          = 1,
                   workers     = 1,
                   seed        = cfg['SEED'],
                   epochs      = cfg['N2V_EPOCHS'])
    emb = np.zeros((num_nodes, cfg['N2V_DIM']), dtype=np.float32)
    for idx in range(num_nodes):
        key = str(idx)
        if key in w2v.wv:
            emb[idx] = w2v.wv[key]
    return emb


def inductive_n2v(X_new_sc, X_train_sc, n2v_train, k=10):
    sim = cosine_similarity(X_new_sc, X_train_sc)
    out = np.zeros((len(X_new_sc), n2v_train.shape[1]), dtype=np.float32)
    for i in range(len(X_new_sc)):
        top_k = np.argpartition(sim[i], -k)[-k:]
        w     = np.maximum(sim[i][top_k], 0.0)
        wsum  = w.sum()
        w     = w / wsum if wsum > 1e-9 else np.ones(k) / k
        out[i] = (n2v_train[top_k] * w[:, None]).sum(axis=0)
    return out


def apply_smote(X, y, seed=42):
    smote        = SMOTE(random_state=seed, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X, y)
    return X_res, y_res


print("Graph / Node2Vec / SMOTE utilities loaded.")


Graph / Node2Vec / SMOTE utilities loaded.


In [12]:
# CELL 4 — MODEL DEFINITIONS
# Exact copies from CV notebooks (single-head GAT — stabilised version).

class GCN(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1   = GCNConv(in_ch, hidden_ch)
        self.c2   = GCNConv(hidden_ch, hidden_ch)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index, edge_weight))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index, edge_weight))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


class GAT(torch.nn.Module):
    """Single-head GAT — stabilised variant used in all CV notebooks."""
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        in_ch     = int(in_ch)
        hidden_ch = int(hidden_ch)
        out_ch    = int(out_ch)
        self.c1   = GATConv(in_ch, hidden_ch,
                             heads=1, concat=False,
                             dropout=dropout, add_self_loops=False)
        self.c2   = GATConv(hidden_ch, hidden_ch,
                             heads=1, concat=False,
                             dropout=dropout, add_self_loops=False)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.dropout(x, self.drop, self.training)
        x = F.elu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.elu(self.c2(x, edge_index))
        return self.lin(x)


class GraphSAGE(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1   = SAGEConv(in_ch, hidden_ch)
        self.c2   = SAGEConv(hidden_ch, hidden_ch)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


class MPNN(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1   = NNConv(in_ch, hidden_ch,
                            Linear(1, in_ch * hidden_ch), aggr='mean')
        self.c2   = NNConv(hidden_ch, hidden_ch,
                            Linear(1, hidden_ch * hidden_ch), aggr='mean')
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def _ea(self, ew, ei, device):
        if ew is not None:
            return ew.unsqueeze(-1)
        return torch.ones(ei.size(1), 1, device=device)

    def forward(self, x, edge_index, edge_weight=None):
        ea = self._ea(edge_weight, edge_index, x.device)
        x  = F.relu(self.c1(x, edge_index, ea))
        x  = F.dropout(x, self.drop, self.training)
        x  = F.relu(self.c2(x, edge_index, ea))
        x  = F.dropout(x, self.drop, self.training)
        return self.lin(x)


class GIN(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1 = GINConv(Sequential(
            Linear(in_ch, hidden_ch), BatchNorm1d(hidden_ch),
            ReLU(), Linear(hidden_ch, hidden_ch)
        ), train_eps=True)
        self.c2 = GINConv(Sequential(
            Linear(hidden_ch, hidden_ch), BatchNorm1d(hidden_ch),
            ReLU(), Linear(hidden_ch, hidden_ch)
        ), train_eps=True)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout

    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


MODEL_REGISTRY = {
    'GCN'       : GCN,
    'GAT'       : GAT,
    'GRAPHSAGE' : GraphSAGE,
    'MPNN'      : MPNN,
    'GIN'       : GIN,
}

def get_model(name, in_ch, hidden_ch, out_ch, dropout):
    return MODEL_REGISTRY[name.upper()](in_ch, hidden_ch, out_ch, dropout)

print("Model definitions loaded: GCN | GAT | GraphSAGE | MPNN | GIN")


Model definitions loaded: GCN | GAT | GraphSAGE | MPNN | GIN


In [13]:
# CELL 5 — DATA LOADING AND INNER SPLIT CONSTRUCTION

df = pd.read_csv(BASE_CONFIG['DATA_PATH'])
print(f"Loaded  : {df.shape[0]} rows × {df.shape[1]} columns")

y_full        = df[BASE_CONFIG['TARGET_COL']].values.astype(np.int64)
X_full        = df.drop(columns=[BASE_CONFIG['TARGET_COL']]).values.astype(np.float32)
feature_names = df.drop(columns=[BASE_CONFIG['TARGET_COL']]).columns.tolist()

print(f"Features: {X_full.shape[1]}")
print(f"PCOS=0  : {(y_full==0).sum()}   PCOS=1  : {(y_full==1).sum()}")

# ── Outer split: reconstruct training pool (test set SEALED, never used) ──
X_train_pool, _X_test, y_train_pool, _y_test = train_test_split(
    X_full, y_full,
    test_size    = BASE_CONFIG['TEST_SIZE'],
    stratify     = y_full,
    random_state = BASE_CONFIG['SEED'],
)
print(f"\nOuter training pool : {len(X_train_pool)} patients  "
      f"(PCOS=0: {(y_train_pool==0).sum()}  PCOS=1: {(y_train_pool==1).sum()})")
print("Test set SEALED — not used in this notebook.")

# ── Inner split: Optuna validation surface ────────────────────────
X_itr, X_ival, y_itr, y_ival = train_test_split(
    X_train_pool, y_train_pool,
    test_size    = BASE_CONFIG['INNER_VAL'],
    stratify     = y_train_pool,
    random_state = BASE_CONFIG['SEED'],
)
print(f"\nInner train : {len(X_itr)} patients  "
      f"(PCOS=0: {(y_itr==0).sum()}  PCOS=1: {(y_itr==1).sum()})")
print(f"Inner val   : {len(X_ival)} patients  "
      f"(PCOS=0: {(y_ival==0).sum()}  PCOS=1: {(y_ival==1).sum()})")
print("\nInner split construction complete.")


Loaded  : 541 rows × 49 columns
Features: 48
PCOS=0  : 364   PCOS=1  : 177

Outer training pool : 432 patients  (PCOS=0: 291  PCOS=1: 141)
Test set SEALED — not used in this notebook.

Inner train : 345 patients  (PCOS=0: 232  PCOS=1: 113)
Inner val   : 87 patients  (PCOS=0: 59  PCOS=1: 28)

Inner split construction complete.


In [14]:
# CELL 6 — SINGLE-TRIAL OBJECTIVE FUNCTION

def run_trial(model_name: str,
              hidden_dim:   int,
              dropout:      float,
              lr:           float,
              weight_decay: float,
              cfg:          dict,
              device) -> float:
    """
    Build the full pipeline for one Optuna trial on the inner split.
    Returns MCC on the inner validation set.
    A return value of -1.0 signals a degenerate (collapsed) trial.
    """
    set_seed(cfg['SEED'])
    n_clinical = X_itr.shape[1]

    # ── Scale ──────────────────────────────────────────────────────
    scaler  = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_itr)
    X_va_sc = scaler.transform(X_ival)

    # ── Build graph from inner-train nodes ─────────────────────────
    ei_tr, ew_tr = build_knn_graph(X_tr_sc, k=cfg['K_NEIGHBOURS'])
    n_real = len(X_tr_sc)

    # ── Node2Vec ───────────────────────────────────────────────────
    n2v_tr   = train_node2vec(ei_tr, n_real, cfg, device)
    X_tr_full = np.concatenate([X_tr_sc, n2v_tr], axis=1)

    # ── SMOTE (on inner-train only) ────────────────────────────────
    X_tr_sm, y_tr_sm = apply_smote(X_tr_full, y_itr, cfg['SEED'])
    n_syn       = len(X_tr_sm) - n_real
    n_train_aug = len(X_tr_sm)

    # ── Add synthetic nodes to graph ───────────────────────────────
    if n_syn > 0:
        X_syn_clin = X_tr_sm[n_real:, :n_clinical]
        ei_aug, ew_aug = add_synthetic_nodes(
            ei_tr, ew_tr, X_tr_sc, X_syn_clin,
            k=cfg['K_NEIGHBOURS'])
    else:
        ei_aug, ew_aug = ei_tr, ew_tr

    # ── Inductive N2V for validation nodes ─────────────────────────
    n2v_va   = inductive_n2v(X_va_sc, X_tr_sc, n2v_tr, k=cfg['K_NEIGHBOURS'])
    X_va_full = np.concatenate([X_va_sc, n2v_va], axis=1)

    # ── Add validation nodes to graph ──────────────────────────────
    ei_full, ew_full = add_val_nodes(
        ei_aug, ew_aug, X_tr_sc, X_va_sc,
        k=cfg['K_NEIGHBOURS'], n_train_aug=n_train_aug)

    # ── Build PyG Data object ──────────────────────────────────────
    X_all  = np.vstack([X_tr_sm, X_va_full])
    y_all  = np.concatenate([y_tr_sm,
                              np.full(len(X_ival), -1, dtype=np.int64)])
    # Validation labels stored separately; -1 as placeholder in y_all
    y_full_arr = np.concatenate([y_tr_sm, y_ival])

    n_total    = len(X_all)
    train_mask = torch.zeros(n_total, dtype=torch.bool)
    val_mask   = torch.zeros(n_total, dtype=torch.bool)
    train_mask[:n_train_aug]            = True
    val_mask[n_train_aug:n_train_aug + len(X_ival)] = True

    data = Data(
        x           = torch.tensor(X_all,          dtype=torch.float),
        edge_index  = ei_full,
        edge_weight = ew_full,
        y           = torch.tensor(y_full_arr,      dtype=torch.long),
        train_mask  = train_mask,
        val_mask    = val_mask,
    ).to(device)

    # ── Class weights ──────────────────────────────────────────────
    n_neg = (y_tr_sm == 0).sum()
    n_pos = (y_tr_sm == 1).sum()
    w_neg = len(y_tr_sm) / (2.0 * n_neg)
    w_pos = len(y_tr_sm) / (2.0 * n_pos)
    cw    = torch.tensor([w_neg, w_pos], dtype=torch.float, device=device)

    # ── Model + optimiser ──────────────────────────────────────────
    in_channels = n_clinical + cfg['N2V_DIM']
    model       = get_model(model_name, in_channels,
                             hidden_dim, 2, dropout).to(device)
    optimiser   = torch.optim.Adam(model.parameters(),
                                    lr=lr, weight_decay=weight_decay)
    scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
                      optimiser, mode='min', factor=0.5,
                      patience=10, min_lr=1e-6)
    criterion   = torch.nn.CrossEntropyLoss(weight=cw)

    # ── Training loop with early stopping on val MCC ───────────────
    best_mcc      = -2.0
    patience_ctr  = 0

    for epoch in range(1, cfg['TRIAL_EPOCHS'] + 1):
        # train step
        model.train()
        optimiser.zero_grad()
        out  = model(data.x, data.edge_index, data.edge_weight)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        if torch.isnan(loss):
            return -1.0
        loss.backward()
        optimiser.step()
        scheduler.step(loss)

        # val evaluation every 5 epochs to keep trial fast
        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                out_val = model(data.x, data.edge_index, data.edge_weight)
                preds   = out_val[data.val_mask].argmax(dim=1).cpu().numpy()
                true    = data.y[data.val_mask].cpu().numpy()

            # Collapse detection: all-same prediction
            if len(np.unique(preds)) == 1:
                patience_ctr += 1
                if patience_ctr >= 3:   # three consecutive checks collapsed
                    return -1.0
                continue

            patience_ctr = 0
            mcc = matthews_corrcoef(true, preds)
            if mcc > best_mcc:
                best_mcc = mcc
            elif epoch > 30 and (mcc < best_mcc - 0.15):
                break   # early stopping: MCC diverging

    # Clamp to [-1, 1]; return -1 if never improved from init
    return float(np.clip(best_mcc, -1.0, 1.0))


print("Trial function defined.")
print("Collapse detection active: three consecutive all-same-prediction")
print("checks within a trial will prune it and return MCC = -1.0")


Trial function defined.
Collapse detection active: three consecutive all-same-prediction
checks within a trial will prune it and return MCC = -1.0


In [15]:
# CELL 7 — OPTUNA OBJECTIVE WRAPPER AND SEARCH SPACE

def make_objective(model_name: str, cfg: dict, device):
    """
    Returns an Optuna objective function for the given model.
    Search space:
      hidden_dim   : {32, 64, 128}
      dropout      : uniform [0.10, 0.50]
      lr           : log-uniform [1e-4, 1e-2]
      weight_decay : log-uniform [1e-5, 1e-3]
    """
    def objective(trial: optuna.Trial) -> float:
        hidden_dim   = trial.suggest_categorical('hidden_dim',   [32, 64, 128])
        dropout      = trial.suggest_float(      'dropout',       0.10,  0.50)
        lr           = trial.suggest_float(      'lr',            1e-4,  1e-2,
                                                  log=True)
        weight_decay = trial.suggest_float(      'weight_decay',  1e-5,  1e-3,
                                                  log=True)

        return run_trial(
            model_name   = model_name,
            hidden_dim   = hidden_dim,
            dropout      = dropout,
            lr           = lr,
            weight_decay = weight_decay,
            cfg          = cfg,
            device       = device,
        )
    return objective


print("Optuna objective wrapper defined.")
print()
print("Search space summary:")
print("  hidden_dim   : categorical  {32, 64, 128}")
print("  dropout      : uniform      [0.10, 0.50]")
print("  lr           : log-uniform  [1e-4, 1e-2]")
print("  weight_decay : log-uniform  [1e-5, 1e-3]")


Optuna objective wrapper defined.

Search space summary:
  hidden_dim   : categorical  {32, 64, 128}
  dropout      : uniform      [0.10, 0.50]
  lr           : log-uniform  [1e-4, 1e-2]
  weight_decay : log-uniform  [1e-5, 1e-3]


In [16]:
# CELL 8 — RUN OPTUNA SEARCH FOR ALL FIVE MODELS
# Expected runtime: ~10–20 min per model on Kaggle GPU T4
# Total: ~60–90 minutes for all five.
#
# Progress is printed after every trial.
# Results are saved incrementally so a kernel crash does not lose work.

MODELS = ['GCN', 'GAT', 'GRAPHSAGE', 'MPNN', 'GIN']

all_study_results = {}
OUT = BASE_CONFIG['OUTPUT_DIR']

for model_name in MODELS:
    print(f"\n{'='*64}")
    print(f"  Optuna search: {model_name}   ({BASE_CONFIG['N_TRIALS']} trials)")
    print(f"{'='*64}")

    study = optuna.create_study(
        direction     = 'maximize',
        study_name    = f'pcos_{model_name.lower()}',
        sampler       = optuna.samplers.TPESampler(seed=GLOBAL_SEED),
        pruner        = optuna.pruners.MedianPruner(n_warmup_steps=10),
    )

    # Seed the study with the original fixed hyperparameters as trial 0
    # so we can always compare against the baseline.
    study.enqueue_trial({
        'hidden_dim'   : 64,
        'dropout'      : 0.3,
        'lr'           : 1e-3,
        'weight_decay' : 1e-4,
    })

    completed = [0]   # mutable counter for callback

    def progress_callback(study, trial):
        completed[0] += 1
        if trial.value is not None:
            status = f"MCC={trial.value:.4f}"
        else:
            status = "pruned"
        best = study.best_value if len(study.trials) > 0 else float('nan')
        print(f"  Trial {completed[0]:>3}/{BASE_CONFIG['N_TRIALS']}  "
              f"{status}   best={best:.4f}   "
              f"params={trial.params}")

    study.optimize(
        make_objective(model_name, BASE_CONFIG, DEVICE),
        n_trials   = BASE_CONFIG['N_TRIALS'],
        callbacks  = [progress_callback],
        gc_after_trial = True,
        show_progress_bar = False,
    )

    best = study.best_trial
    print(f"\n  ✓ {model_name} best trial #{best.number}")
    print(f"    MCC          : {best.value:.4f}")
    print(f"    hidden_dim   : {best.params['hidden_dim']}")
    print(f"    dropout      : {best.params['dropout']:.4f}")
    print(f"    lr           : {best.params['lr']:.6f}")
    print(f"    weight_decay : {best.params['weight_decay']:.6f}")

    all_study_results[model_name] = {
        'best_mcc'   : best.value,
        'best_trial' : best.number,
        'params'     : best.params,
        'all_trials' : [
            {'number': t.number, 'value': t.value, 'params': t.params}
            for t in study.trials if t.value is not None
        ],
    }

    # Save incrementally after each model
    out_path = os.path.join(OUT, 'best_hyperparams.json')
    with open(out_path, 'w') as f:
        json.dump(all_study_results, f, indent=2)
    print(f"  Saved -> {out_path}")

print(f"\n{'='*64}")
print("  Optuna search complete for all five models.")
print(f"{'='*64}")



  Optuna search: GCN   (100 trials)
  Trial   1/100  MCC=0.6908   best=0.6908   params={'hidden_dim': 64, 'dropout': 0.3, 'lr': 0.001, 'weight_decay': 0.0001}
  Trial   2/100  MCC=0.5984   best=0.6908   params={'hidden_dim': 64, 'dropout': 0.3394633936788146, 'lr': 0.0002051338263087451, 'weight_decay': 2.0511104188433963e-05}
  Trial   3/100  MCC=0.5984   best=0.6908   params={'hidden_dim': 64, 'dropout': 0.3832290311184182, 'lr': 0.00010994335574766199, 'weight_decay': 0.0008706020878304854}
  Trial   4/100  MCC=0.6390   best=0.6908   params={'hidden_dim': 32, 'dropout': 0.17336180394137352, 'lr': 0.0004059611610484307, 'weight_decay': 0.00011207606211860574}
  Trial   5/100  MCC=0.7501   best=0.7501   params={'hidden_dim': 128, 'dropout': 0.15579754426081674, 'lr': 0.0003839629299804173, 'weight_decay': 5.4041038546473305e-05}
  Trial   6/100  MCC=0.6991   best=0.7501   params={'hidden_dim': 64, 'dropout': 0.3056937753654446, 'lr': 0.0015304852121831463, 'weight_decay': 1.238513729

In [17]:
# CELL 9 — RESULTS SUMMARY TABLE

print("\n" + "="*72)
print(f"  {'Model':<12}  {'Best MCC':>10}  {'hidden_dim':>12}  "
      f"{'dropout':>9}  {'lr':>10}  {'weight_decay':>13}")
print("="*72)

baseline = {
    'hidden_dim'   : 64,
    'dropout'      : 0.3,
    'lr'           : 1e-3,
    'weight_decay' : 1e-4,
}

for model_name in MODELS:
    r = all_study_results[model_name]
    p = r['params']
    print(f"  {model_name:<12}  {r['best_mcc']:>10.4f}  "
          f"{p['hidden_dim']:>12}  {p['dropout']:>9.4f}  "
          f"{p['lr']:>10.6f}  {p['weight_decay']:>13.6f}")

print("="*72)
print()
print("Baseline (uniform config used in original CV notebooks):")
print(f"  hidden_dim={baseline['hidden_dim']}  dropout={baseline['dropout']}  "
      f"lr={baseline['lr']}  weight_decay={baseline['weight_decay']}")
print()
print("Next step:")
print("  Update CONFIG in each model's CV notebook with per-model params above,")
print("  re-run the 10-fold CV, and compare MCC against the baseline run.")
print()
print(f"Full results saved to: {os.path.join(BASE_CONFIG['OUTPUT_DIR'], 'best_hyperparams.json')}")



  Model           Best MCC    hidden_dim    dropout          lr   weight_decay
  GCN               0.8177            32     0.3411    0.003183       0.000068
  GAT               0.8199           128     0.4101    0.007568       0.000616
  GRAPHSAGE         0.7943            32     0.4913    0.006033       0.000557
  MPNN              0.8461           128     0.3461    0.002310       0.000130
  GIN               0.7511            32     0.4796    0.008536       0.000414

Baseline (uniform config used in original CV notebooks):
  hidden_dim=64  dropout=0.3  lr=0.001  weight_decay=0.0001

Next step:
  Update CONFIG in each model's CV notebook with per-model params above,
  re-run the 10-fold CV, and compare MCC against the baseline run.

Full results saved to: /kaggle/working/best_hyperparams.json


In [18]:
# CELL 10 — HOW TO LOAD BEST HYPERPARAMETERS IN YOUR CV NOTEBOOKS
#
# Add the following block at the top of each CV notebook (after the
# BASE_CONFIG definition) to replace the fixed values with Optuna's
# per-model results.
#
# Example for the GraphSAGE notebook:
#
#   import json
#   MODEL_NAME = 'GRAPHSAGE'
#   with open('/kaggle/input/.../best_hyperparams.json') as f:
#       optuna_params = json.load(f)
#   best = optuna_params[MODEL_NAME]['params']
#
#   CONFIG['HIDDEN_DIM']   = best['hidden_dim']
#   CONFIG['DROPOUT']      = best['dropout']
#   CONFIG['LR']           = best['lr']
#   CONFIG['WEIGHT_DECAY'] = best['weight_decay']
#
#   print(f"Loaded Optuna hyperparameters for {MODEL_NAME}:")
#   for k, v in best.items():
#       print(f"  {k}: {v}")
#
# The rest of the notebook runs unchanged.

# ── Verify the saved JSON is readable ─────────────────────────────
out_path = os.path.join(BASE_CONFIG['OUTPUT_DIR'], 'best_hyperparams.json')
with open(out_path) as f:
    loaded = json.load(f)

print("Verification — best_hyperparams.json contents:")
for model, result in loaded.items():
    print(f"\n  {model}")
    print(f"    best_mcc : {result['best_mcc']:.4f}")
    for k, v in result['params'].items():
        print(f"    {k:<14}: {v}")


Verification — best_hyperparams.json contents:

  GCN
    best_mcc : 0.8177
    hidden_dim    : 32
    dropout       : 0.3411107246264348
    lr            : 0.0031830107495889903
    weight_decay  : 6.75766430394288e-05

  GAT
    best_mcc : 0.8199
    hidden_dim    : 128
    dropout       : 0.4100531293444458
    lr            : 0.007568292060167618
    weight_decay  : 0.0006161049539380963

  GRAPHSAGE
    best_mcc : 0.7943
    hidden_dim    : 32
    dropout       : 0.49129425695052514
    lr            : 0.006032530190428526
    weight_decay  : 0.0005570996243998471

  MPNN
    best_mcc : 0.8461
    hidden_dim    : 128
    dropout       : 0.3461096359526714
    lr            : 0.002310345028973248
    weight_decay  : 0.00013016947717264555

  GIN
    best_mcc : 0.7511
    hidden_dim    : 32
    dropout       : 0.4795542149013333
    lr            : 0.00853618986286683
    weight_decay  : 0.0004138040112561013
